# Hand Landmarks Detection with MediaPipe Tasks
**Local version** — works outside Google Colab, includes live webcam feed.

## Step 1: Install dependencies

In [1]:
%pip install mediapipe opencv-python

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\aphan\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Step 2: Download the model

In [2]:
import urllib.request
import os

MODEL_URL = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
MODEL_PATH = "hand_landmarker.task"

if not os.path.exists(MODEL_PATH):
    print("Downloading model...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("Done!")
else:
    print("Model already downloaded.")

Model already downloaded.


## Step 3: Visualization utilities
Uses only `cv2` and `numpy` — no mediapipe internal drawing imports needed.

In [8]:
import cv2
import numpy as np

MARGIN = 10
FONT_SIZE = 1
FONT_THICKNESS = 1
HANDEDNESS_TEXT_COLOR = (88, 205, 54)

# 21-landmark hand skeleton connections
HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),
    (5,9),(9,13),(13,17)
]

# Landmark indices
# Fingertips:  thumb=4, index=8, middle=12, ring=16, pinky=20
# PIP joints:  index=6, middle=10, ring=14, pinky=18
# MCP joints:  index=5, middle=9,  ring=13, pinky=17
# Thumb IP:    3

def is_finger_extended(landmarks, tip_idx, pip_idx):
    # Returns True if fingertip is above (lower y value) its PIP joint
    return landmarks[tip_idx].y < landmarks[pip_idx].y

def is_thumb_extended(landmarks, handedness_label):
    # Thumb uses x-axis: tip further from palm than IP joint
    tip_x  = landmarks[4].x
    ip_x   = landmarks[3].x
    # For right hand, tip should be to the left (smaller x); flip for left
    if handedness_label == "Right":
        return tip_x < ip_x
    else:
        return tip_x > ip_x

def classify_gesture(landmarks, handedness_label):
    thumb  = is_thumb_extended(landmarks, handedness_label)
    index  = is_finger_extended(landmarks, 8, 6)
    middle = is_finger_extended(landmarks, 12, 10)
    ring   = is_finger_extended(landmarks, 16, 14)
    pinky  = is_finger_extended(landmarks, 20, 18)

    extended = [index, middle, ring, pinky]
    num_extended = sum(extended)

    if num_extended == 0:
        return "Rock", (50, 50, 220)       # Blue
    elif num_extended == 4:
        return "Paper", (220, 50, 50)     # Yellow
    elif index and middle and not ring and not pinky:
        return "Scissors", (50, 220, 50)   # green
    else:
        return "...", (150, 150, 150)      # unknown

def draw_landmarks_on_image(rgb_image, detection_result):
    hand_landmarks_list = detection_result.hand_landmarks
    handedness_list = detection_result.handedness
    annotated_image = np.copy(rgb_image)
    height, width, _ = annotated_image.shape

    for idx in range(len(hand_landmarks_list)):
        hand_landmarks = hand_landmarks_list[idx]
        handedness = handedness_list[idx]
        handedness_label = handedness[0].category_name

        points = [
            (int(lm.x * width), int(lm.y * height))
            for lm in hand_landmarks
        ]

        # Classify gesture
        gesture, gesture_color = classify_gesture(hand_landmarks, handedness_label)

        # Draw connections
        for start, end in HAND_CONNECTIONS:
            cv2.line(annotated_image, points[start], points[end], (200, 200, 200), 2)

        # Draw landmark dots
        for i, pt in enumerate(points):
            color = (0, 128, 255) if i % 4 == 0 else (255, 255, 255)
            cv2.circle(annotated_image, pt, 5, color, -1)
            cv2.circle(annotated_image, pt, 5, (0, 0, 0), 1)

        x_coords = [p[0] for p in points]
        y_coords = [p[1] for p in points]
        text_x = min(x_coords)
        text_y = max(30, min(y_coords) - MARGIN)

        # Draw handedness label
        cv2.putText(annotated_image, handedness_label,
                    (text_x, text_y - 30), cv2.FONT_HERSHEY_DUPLEX,
                    0.7, (88, 205, 54), 1, cv2.LINE_AA)

        # Draw gesture label (bigger, coloured)
        cv2.putText(annotated_image, gesture,
                    (text_x, text_y), cv2.FONT_HERSHEY_DUPLEX,
                    1.2, gesture_color, 2, cv2.LINE_AA)

    return annotated_image

print("Utilities ready.")


Utilities ready.


## Step 4: Live webcam feed

In [9]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2
import numpy as np
import threading
import time

CAMERA_INDEX = 1 # Webcam

# Shared result between callback thread and display loop
latest_result = {"result": None}
result_lock = threading.Lock()

def result_callback(result, output_image, timestamp_ms):
    with result_lock:
        latest_result["result"] = result

BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.LIVE_STREAM,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    result_callback=result_callback
)

# CAP_DSHOW avoids Windows camera timeout issues
cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)

if not cap.isOpened():
    print(f"ERROR: Could not open camera index {CAMERA_INDEX}.")
    print("Run the Find your camera index cell above first.")
else:
    # Set buffer size to 1 so we always get the latest frame
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)

    # --- Warm-up: drain the first ~30 frames while the sensor stabilises ---
    print("Warming up camera...", end="", flush=True)
    for _ in range(30):
        cap.read()
    print(" ready!")

    print(f"Camera {CAMERA_INDEX} opened — showing window. Press Q to quit.")
    with HandLandmarker.create_from_options(options) as landmarker:
        consecutive_failures = 0
        while True:
            ret, frame = cap.read()

            if not ret or frame is None:
                consecutive_failures += 1
                print(f"Frame grab failed (attempt {consecutive_failures}/10)...")
                if consecutive_failures >= 10:
                    print("Too many consecutive failures — giving up.")
                    break
                time.sleep(0.05)   # brief pause then retry
                continue

            consecutive_failures = 0  # reset on success

            # MediaPipe expects RGB; OpenCV gives BGR
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

            # Timestamp must be monotonically increasing (milliseconds)
            timestamp_ms = int(time.time() * 1000)
            landmarker.detect_async(mp_image, timestamp_ms)

            with result_lock:
                result = latest_result["result"]

            if result is not None:
                annotated = draw_landmarks_on_image(rgb_frame, result)
                display_frame = cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR)
            else:
                display_frame = frame

            cv2.imshow("Hand Landmarks - Live (press Q to quit)", display_frame)

            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

    cap.release()
    cv2.destroyAllWindows()
    print("Webcam released.")


Warming up camera... ready!
Camera 1 opened — showing window. Press Q to quit.
Webcam released.
